In [64]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import numpy as np
import pandas as pd
from data.market_data import get_multiple_tickers
from pricing.american import american_put_without_dividends, american_put_with_dividends, american_call_without_dividends, american_call_with_dividends
from pricing.european import heston_european_call_option, heston_european_put_option
from calibration.implied_vol import implied_volatility
from calibration.calibrate_heston import calibrate_heston


##### tickers and market constraint

In [65]:
tickers = ['AAPL', 'NVDA']
Ns = 40
Nv = 20
Nt = 40
r = 0.05
q=0
N= 10000
M = 100
spread_limit = 0.05

In [66]:
options_df = get_multiple_tickers(tickers, spread_limit, r, q)

pulling...AAPL...
pulling...NVDA...


In [67]:
options_df.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,spot,ticker,ExerciseStyle,mid_price,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound
0,AAPL260323C00247500,2026-03-20 19:59:48+00:00,247.5,2.21,2.19,2.30,-1.040000,-32.000000,5733.0,390.0,...,247.990005,AAPL,american,2.245,0.005811,0.048998,0.998024,247.990005,247.428101,0.561905
1,AAPL260323P00247500,2026-03-20 19:59:59+00:00,247.5,1.62,1.54,1.60,-0.130000,-7.428571,12963.0,2860.0,...,247.990005,AAPL,american,1.570,0.005811,0.038217,0.998024,247.990005,247.428101,0.000000
2,AAPL260325C00245000,2026-03-20 19:42:56+00:00,245.0,4.10,4.95,5.20,-1.800000,-30.508476,504.0,68.0,...,247.990005,AAPL,american,5.075,0.011290,0.049261,0.987943,247.990005,244.861732,3.128273
3,AAPL260325C00250000,2026-03-20 19:59:42+00:00,250.0,2.18,2.10,2.20,-0.710000,-24.567474,9288.0,515.0,...,247.990005,AAPL,american,2.150,0.011290,0.046512,1.008105,247.990005,249.858911,0.000000
4,AAPL260327C00230000,2026-03-20 19:58:53+00:00,230.0,18.53,18.25,18.85,-0.529999,-2.780686,38.0,37.0,...,247.990005,AAPL,american,18.550,0.016770,0.032345,0.927457,247.990005,229.807228,18.182777


In [103]:
def price_row(row, r, q, heston_params ):
    S0 = row['spot']
    K=row['strike']
    T= row['T']
    option_type = row['type']

    v0, kappa, theta, sigma, rho = heston_params

    if option_type == 'call':
        if q==0:
            #print('american_call_without_dividends ....exexcuted')
            return american_call_without_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho)
        else:
            #print('american_call_with_dividends....exexcuted')
            return american_call_with_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N, q)
    elif option_type == 'put':
        if q==0:
            #print('american_put_without_dividends....exexcuted')
            return american_put_without_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N)
        else:
            #print('american_put_with_dividends....exexcuted')
            return american_put_with_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N, q)

#### Heston Model Calibration 
###### Add mid price

In [69]:
calibration_df_nvda = options_df[options_df['ticker']=='NVDA'].copy()

calibration_df_nvda.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,spot,ticker,ExerciseStyle,mid_price,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound
427,NVDA260323C00167500,2026-03-20 19:59:36+00:00,167.5,5.75,5.75,5.90,-5.90,-50.643776,295.0,36.0,...,172.699997,NVDA,american,5.825,0.005811,0.025751,0.969890,172.699997,167.451342,5.248655
428,NVDA260323C00170000,2026-03-20 19:59:57+00:00,170.0,3.70,3.75,3.80,-5.00,-57.471264,3976.0,238.0,...,172.699997,NVDA,american,3.775,0.005811,0.013245,0.984366,172.699997,169.950615,2.749381
429,NVDA260323C00172500,2026-03-20 19:59:53+00:00,172.5,2.06,2.06,2.08,-4.09,-66.504070,10011.0,591.0,...,172.699997,NVDA,american,2.070,0.005811,0.009662,0.998842,172.699997,172.449889,0.250108
430,NVDA260323C00175000,2026-03-20 20:00:00+00:00,175.0,0.90,0.89,0.91,-3.90,-81.250000,48364.0,4801.0,...,172.699997,NVDA,american,0.900,0.005811,0.022222,1.013318,172.699997,174.949163,0.000000
431,NVDA260323C00177500,2026-03-20 19:59:59+00:00,177.5,0.31,0.31,0.32,-2.58,-89.273360,47453.0,4761.0,...,172.699997,NVDA,american,0.315,0.005811,0.031746,1.027794,172.699997,177.448437,0.000000


In [70]:
calibration_df_nvda.shape

(698, 26)

In [13]:
initial_guess = [
    0.04,   # v0  (initial variance ≈ 20% vol)
    2.0,    # kappa
    0.04,   # theta
    0.5,    # sigma
    -0.7    # rho
]

In [15]:
options_df_nvda['heston_price'] = options_df_nvda.apply(lambda row: price_row(row, r, q, heston_params), axis=1)

american_call_without_dividends ....exexcuted


In [16]:
print("Calibrated Parameters:")
print("v0    =", params_opt[0])
print("kappa =", params_opt[1])
print("theta =", params_opt[2])
print("sigma =", params_opt[3])
print("rho   =", params_opt[4])
print("\nFinal Loss =", loss_val)

Calibrated Parameters:
v0    = 0.24403566968414625
kappa = 2.000169494947594
theta = 0.1087414325971798
sigma = 0.5085892157308337
rho   = -0.7084200062062825

Final Loss = 0.03448939252321163


In [71]:
calibrated_heston_params = (0.24403566968414625, 2.000169494947594, 0.1087414325971798, 0.5085892157308337, -0.7084200062062825)

In [72]:
options_df_nvda = options_df[options_df['ticker']=='NVDA'].copy()
options_df_nvda['calibrated_heston_price'] = options_df_nvda.apply(lambda row: price_row(row, r, q, calibrated_heston_params), axis=1)
options_df_nvda.head()

/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,ticker,ExerciseStyle,mid_price,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound,calibrated_heston_price
427,NVDA260323C00167500,2026-03-20 19:59:36+00:00,167.5,5.75,5.75,5.90,-5.90,-50.643776,295.0,36.0,...,NVDA,american,5.825,0.005811,0.025751,0.969890,172.699997,167.451342,5.248655,6.009454
428,NVDA260323C00170000,2026-03-20 19:59:57+00:00,170.0,3.70,3.75,3.80,-5.00,-57.471264,3976.0,238.0,...,NVDA,american,3.775,0.005811,0.013245,0.984366,172.699997,169.950615,2.749381,4.187850
429,NVDA260323C00172500,2026-03-20 19:59:53+00:00,172.5,2.06,2.06,2.08,-4.09,-66.504070,10011.0,591.0,...,NVDA,american,2.070,0.005811,0.009662,0.998842,172.699997,172.449889,0.250108,2.715052
430,NVDA260323C00175000,2026-03-20 20:00:00+00:00,175.0,0.90,0.89,0.91,-3.90,-81.250000,48364.0,4801.0,...,NVDA,american,0.900,0.005811,0.022222,1.013318,172.699997,174.949163,0.000000,1.621934
431,NVDA260323C00177500,2026-03-20 19:59:59+00:00,177.5,0.31,0.31,0.32,-2.58,-89.273360,47453.0,4761.0,...,NVDA,american,0.315,0.005811,0.031746,1.027794,172.699997,177.448437,0.000000,0.885513


In [77]:
options_df_nvda_vol = options_df_nvda.copy()
options_df_nvda_vol.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,ticker,ExerciseStyle,mid_price,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound,calibrated_heston_price
427,NVDA260323C00167500,2026-03-20 19:59:36+00:00,167.5,5.75,5.75,5.90,-5.90,-50.643776,295.0,36.0,...,NVDA,american,5.825,0.005811,0.025751,0.969890,172.699997,167.451342,5.248655,6.009454
428,NVDA260323C00170000,2026-03-20 19:59:57+00:00,170.0,3.70,3.75,3.80,-5.00,-57.471264,3976.0,238.0,...,NVDA,american,3.775,0.005811,0.013245,0.984366,172.699997,169.950615,2.749381,4.187850
429,NVDA260323C00172500,2026-03-20 19:59:53+00:00,172.5,2.06,2.06,2.08,-4.09,-66.504070,10011.0,591.0,...,NVDA,american,2.070,0.005811,0.009662,0.998842,172.699997,172.449889,0.250108,2.715052
430,NVDA260323C00175000,2026-03-20 20:00:00+00:00,175.0,0.90,0.89,0.91,-3.90,-81.250000,48364.0,4801.0,...,NVDA,american,0.900,0.005811,0.022222,1.013318,172.699997,174.949163,0.000000,1.621934
431,NVDA260323C00177500,2026-03-20 19:59:59+00:00,177.5,0.31,0.31,0.32,-2.58,-89.273360,47453.0,4761.0,...,NVDA,american,0.315,0.005811,0.031746,1.027794,172.699997,177.448437,0.000000,0.885513


#### Market IV

In [78]:
options_df_nvda_vol["market_iv"] = options_df_nvda_vol.apply(
    lambda row: implied_volatility(
        row["mid_price"],
        row["spot"],
        row["strike"],
        r,
        row["T"],
        row["type"],
        q
    ),
    axis=1
)
options_df_nvda_vol.head()

contractSymbol             lastTradeDate  strike  lastPrice   bid  \
427  NVDA260323C00167500 2026-03-20 19:59:36+00:00   167.5       5.75  5.75   
428  NVDA260323C00170000 2026-03-20 19:59:57+00:00   170.0       3.70  3.75   
429  NVDA260323C00172500 2026-03-20 19:59:53+00:00   172.5       2.06  2.06   
430  NVDA260323C00175000 2026-03-20 20:00:00+00:00   175.0       0.90  0.89   
431  NVDA260323C00177500 2026-03-20 19:59:59+00:00   177.5       0.31  0.31   

      ask  change  percentChange   volume  openInterest  ...  ExerciseStyle  \
427  5.90   -5.90     -50.643776    295.0          36.0  ...       american   
428  3.80   -5.00     -57.471264   3976.0         238.0  ...       american   
429  2.08   -4.09     -66.504070  10011.0         591.0  ...       american   
430  0.91   -3.90     -81.250000  48364.0        4801.0  ...       american   
431  0.32   -2.58     -89.273360  47453.0        4761.0  ...       american   

     mid_price         T rel_spread moneyness forward_spot  disc_strike  \
427      5.825  0.005811   0.025751  0.969890   172.699997   167.451342   
428      3.775  0.005811   0.013245  0.984366   172.699997   169.950615   
429      2.070  0.005811   0.009662  0.998842   172.699997   172.449889   
430      0.900  0.005811   0.022222  1.013318   172.699997   174.949163   
431      0.315  0.005811   0.031746  1.027794   172.699997   177.448437   

    lower_bound calibrated_heston_price  market_iv  
427    5.248655                6.009454   0.447166  
428    2.749381                4.187850   0.407523  
429    0.250108                2.715052   0.370121  
430    0.000000                1.621934   0.341696  
431    0.000000                0.885513   0.329873  

[5 rows x 28 columns]

##### Volatility diagnostic

In [76]:
options_df_nvda_vol['iv_diag'].value_counts(dropna=False)

KeyError: 'iv_diag'

#### Calibrated Model IV

In [25]:
options_df_nvda_vol["model_iv"] = options_df_nvda_vol.apply(
    lambda row: implied_volatility(
        row["calibrated_heston_price"],
        row["spot"],
        row["strike"],
        r,
        row["T"],
        row["type"],
        q
    ),
    axis=1
)
options_df_nvda.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound,calibrated_heston_price,market_iv,iv_diag,model_iv
1282,NVDA260313C00050000,2026-03-10 15:47:36+00:00,50.0,135.67,130.30,135.9,0.0,0.0,5.0,20.0,...,call,2026-03-13,0.0,183.139999,NVDA,american,133.100,133.313665,NaN,NaN
428,NVDA260323C00170000,2026-03-20 19:59:57+00:00,170.0,3.70,3.75,3.80,-5.00,-57.471264,3976.0,238.0,...,call,2026-03-13,0.0,183.139999,NVDA,american,128.225,127.930029,NaN,NaN
429,NVDA260323C00172500,2026-03-20 19:59:53+00:00,172.5,2.06,2.06,2.08,-4.09,-66.504070,10011.0,591.0,...,call,2026-03-13,0.0,183.139999,NVDA,american,123.150,123.120645,NaN,NaN
430,NVDA260323C00175000,2026-03-20 20:00:00+00:00,175.0,0.90,0.89,0.91,-3.90,-81.250000,48364.0,4801.0,...,call,2026-03-13,0.0,183.139999,NVDA,american,118.150,118.501314,NaN,NaN
431,NVDA260323C00177500,2026-03-20 19:59:59+00:00,177.5,0.31,0.31,0.32,-2.58,-89.273360,47453.0,4761.0,...,call,2026-03-13,0.0,183.139999,NVDA,american,113.200,113.267181,NaN,NaN


In [114]:
from scipy.interpolate import griddata

M = options_df_nvda_vol["moneyness"].values
T = options_df_nvda_vol["T"].values
IV = options_df_nvda_vol["market_iv"].values

# grid
m_grid = np.linspace(M.min(), M.max(), 50)
t_grid = np.linspace(T.min(), T.max(), 50)

M_grid, T_grid = np.meshgrid(m_grid, t_grid)

IV_grid = griddata(
    (M, T),
    IV,
    (M_grid, T_grid),
    method="cubic"
)

In [115]:
import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Surface(
            x=M_grid,
            y=T_grid,
            z=IV_grid,
            colorscale="Jet"
        )
    ]
)

fig.update_layout(
    title="Implied Volatility Surface",
    scene=dict(
        xaxis_title="Moneyness (S/K)",
        yaxis_title="Time to Maturity",
        zaxis_title="Implied Volatility"
    )
)

fig.show()